<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

**GUIDE**

The goal of `fastllm` library which we are currently refactoring module by module here in these dialogs is to create a unified llm api for OpenAI responses, OpenAI chat, Anthropic and Gemini apis. Users should be able to use this unified / canonical library to be able to swap models/providers effortlessly without needing to change any code.

Refactored dialogs so far:

- `aai-ws/fastllm/nbs/00_types` exists is to create a canonical internal representations across model providers so that fastllm can be a unified api where users can swap models without changing code during a chat session. 
- `aai-ws/fastllm/nbs/01_normalize` exists to convert model responses or stream events into normalized internal representations.
- `aai-ws/fastllm/nbs/02_streaming` provides lossless stream collation ([`acollect_stream`](https://AnswerDotAI.github.io/fastllm/streaming.html#acollect_stream)) that collects fragmented Delta streams into `StreamSummary` with a [`Msg`](https://AnswerDotAI.github.io/fastllm/types.html#msg) matching the shape of `Completion.message`. Covers tool call assembly, thinking/text interleaving, and server tool result preservation across all four providers.


You can find the pre-refactor and fully functional snapshot of this library `~/aai-ws/fastllm/fastllm_py/*.py`. The already refactored modules might differ from their counterparts in `fastllm_py/types.py` and `fastllm_py/normalize.py`, as refactoring involves simplifying code, refactoring the code to fastai / literate programming style, redesigning parts, and fixing issues/bugs.

 When users continue the chat with the same provider raw data is used for constructing the provider native input with all required fields to keep the cache intact. When switching over to providers, canonical data is transformed into provider counterpart with best effort.

## Imports

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = Path('../specs/')

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
gem_cli.models

- models.generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a model response given an input `GenerateContentRequest`. Refer to the [text generation guide](https://ai.google.dev/gemini-api/docs/text-generation) for detailed usage information. Input capabilities differ between models, including tuned models. Refer to the [model guide](https://ai.google.dev/gemini-api/docs/models/gemini) and [tuning guide](https://ai.google.dev/gemini-api/docs/model-tuning) for details.*
- models.generate_answer(model, contents, answer_style, inline_passages, semantic_retriever, safety_settings, temperature): *Generates a grounded answer from the model given an input `GenerateAnswerRequest`.*
- models.stream_generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a [streamed response](https://ai.google.dev/gemini-api/docs/text-generation?lang=python#generate-a-text-stream) from the model given an input `GenerateContentRequest`.*
- models.embed_content(model, content, task_type, title, output_dimensionality): *Generates a text embedding vector from the input `Content` using the specified [Gemini Embedding model](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding).*
- models.batch_embed_contents(model, requests): *Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects.*
- models.count_tokens(model, contents, generate_content_request): *Runs a model's tokenizer on input `Content` and returns the token count. Refer to the [tokens guide](https://ai.google.dev/gemini-api/docs/tokens) to learn more about tokens.*
- models.batch_generate_content(model, batch): *Enqueues a batch of `GenerateContent` requests for batch processing.*
- models.async_batch_embed_content(model, batch): *Enqueues a batch of `EmbedContent` requests for batch processing. We have a `BatchEmbedContents` handler in `GenerativeService`, but it was synchronized. So we name this one to be `Async` to avoid confusion.*
- models.generate_message(model, prompt, temperature, candidate_count, top_p, top_k): *Generates a response from the model given an input `MessagePrompt`.*
- models.count_message_tokens(model, prompt): *Runs a model's tokenizer on a string and returns the token count.*
- models.get(name): *Gets information about a specific `Model` such as its version number, token limits, [parameters](https://ai.google.dev/gemini-api/docs/models/generative-models#model-parameters) and other metadata. Refer to the [Gemini models guide](https://ai.google.dev/gemini-api/docs/models/gemini) for detailed model information.*
- models.list(page_size, page_token): *Lists the [`Model`s](https://ai.google.dev/gemini-api/docs/models/gemini) available through the Gemini API.*
- models.predict(model, instances, parameters): *Performs a prediction request.*
- models.predict_long_running(model, instances, parameters): *Same as Predict but returns an LRO.*
- models.generate_text(model, prompt, temperature, candidate_count, max_output_tokens, top_p, top_k, safety_settings, stop_sequences): *Generates a response from the model given an input message.*
- models.embed_text(model, text): *Generates an embedding from the model given an input message.*
- models.batch_embed_text(model, texts, requests): *Generates multiple embeddings from the model given input text in a synchronous call.*
- models.count_text_tokens(model, prompt): *Runs a model's tokenizer on a text and returns the token count.*

## Tool Calls

All four providers represent tool invocations differently in their wire format:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — Tool calls are flat output items: `{type: "function_call", call_id, name, arguments}` where `arguments` is a JSON **string**. Streamed via `response.function_call_arguments.delta` events.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — Tool calls are nested: `{id, type: "function", function: {name, arguments}}` where `arguments` is a JSON **string**. Streamed in chunks via `tool_calls[i].function.arguments` deltas.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — Tool calls are `tool_use` content blocks: `{id, name, input, type: "tool_use"}` where `input` is a parsed JSON object. Streamed via `input_json_delta` events.
- **[Gemini](https://ai.google.dev/api/generate-content)** — Tool calls are `functionCall` parts: `{id, name, args}` where `args` is a parsed JSON object. Not chunked during streaming.

> **Spec sources:** OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347–44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869–34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563–13588`), Gemini `FunctionCall` (`specs/gemini.json:273–293`).

[`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) canonicalizes these into a flat `(id, name, arguments)` triple where `arguments` is always a parsed `dict`:

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini | [`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) |
|---|---|---|---|---|---|
| ID | `call_id` | `id` | `id` | `id` | `id` |
| Name | `name` | `function.name` | `name` | `name` | `name` |
| Args | `arguments` (JSON string) | `function.arguments` (JSON string) | `input` (object) | `args` (object) | `arguments` (dict) |

In [0]:
#| echo: false
#| output: asis
show_doc(ToolCall)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L24){target="_blank" style="float:right; font-size:smaller"}

### ToolCall

```python

def ToolCall(
    id:str, name:str, arguments:dict=<factory>, server:bool=False, extra:dict=<factory>
)->None:


```

*Normalized tool call.*

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### anthropic_tool_calls

In [0]:
#| echo: false
#| output: asis
show_doc(anthropic_tool_calls)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L40){target="_blank" style="float:right; font-size:smaller"}

### anthropic_tool_calls

```python

def anthropic_tool_calls(
    resp, delta:bool=False
):


```

*Extract Anthropic tool_use blocks as normalized tool calls.*

In [0]:
#| echo: false
#| output: asis
show_doc(anthropic_tool_call)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L35){target="_blank" style="float:right; font-size:smaller"}

### anthropic_tool_call

```python

def anthropic_tool_call(
    b
):


```

*Call self as a function.*

User defined:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
anthropic_tool_calls(resp)

[ToolCall(id='toolu_01GQXvetC2hZwQYLTXoSJByr', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_017qg9Kstj9cQycBE3NqR4QZ', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

Web search:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
anthropic_tool_calls(resp)

[ToolCall(id='srvtoolu_011JVe4eamPpdUBKiq934MXd', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, extra={'caller': {'type': 'direct'}})]

MCP:

In [ ]:
anthropic_tool_calls({"content": [
    {"type": "mcp_tool_use", "id": "mcptoolu_abc123", "name": "get_weather",
     "input": {"city": "Istanbul"}, "server_name": "weather-server"}
]})

[ToolCall(id='mcptoolu_abc123', name='get_weather', arguments={'city': 'Istanbul'}, server=True, extra={'server_name': 'weather-server'})]

In [ ]:
resp

{'model': 'claude-sonnet-4-20250514',
 'id': 'msg_012DSP2yz1cXUikugK239NkF',
 'type': 'message',
 'role': 'assistant',
 'content': [{'type': 'server_tool_use',
   'id': 'srvtoolu_011JVe4eamPpdUBKiq934MXd',
   'name': 'web_search',
   'input': {'query': 'Istanbul weather today'},
   'caller': {'type': 'direct'}},
  {'type': 'web_search_tool_result',
   'tool_use_id': 'srvtoolu_011JVe4eamPpdUBKiq934MXd',
   'content': [{'type': 'web_search_result',
     'title': 'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather',
     'url': 'https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251',
     'encrypted_content': 'EsgUCioIDhgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDANLRULrc5btsRO3VhoMIqlddbKSrc06qQWyIjDXIddpkjTB1bTqFZuQ/vjGX5B5+T0L6+b7CMCIIluz776JFvHG3cdhJdN9l/nlumMqyxOApij/OhfYSZGChkBJUcn/Juv4WRjo9Ndr5z9sBbFAkZyg7vbPQvbsRrZVt9yf94JpRr5GZ9UKWl2SVemc5Us3p/n2EjKmrUszdHGNZSurdtxamJmlKxyWy+wKUmHXO9cNEQSUqafWmnTgJ0I8wQw5LksiOreQFlIFUYQjplSRnnYZWX0+k21usts82

### openai_responses_tool_calls

In [0]:
#| echo: false
#| output: asis
show_doc(openai_responses_tool_calls)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L59){target="_blank" style="float:right; font-size:smaller"}

### openai_responses_tool_calls

```python

def openai_responses_tool_calls(
    resp
):


```

*Extract Responses API tool call items as normalized tool calls.*

In [0]:
#| echo: false
#| output: asis
show_doc(openai_responses_tool_call)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L48){target="_blank" style="float:right; font-size:smaller"}

### openai_responses_tool_call

```python

def openai_responses_tool_call(
    item
):


```

*Call self as a function.*

User defined:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
openai_responses_tool_calls(resp)

[ToolCall(id='fc_07e79637b65d510c0069d66ff214c48196b5e751ed4d144a21', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_DAJIJCaYpEhIjo9Uj9payJqN'}),
 ToolCall(id='fc_07e79637b65d510c0069d66ff214d481968f1cc2376401e205', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_GlfRTq3CNf8ltWxNpcGyW5XC'})]

Web search:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is the weather in San Francisco today?",
    tools=[{"type": "web_search_preview"}]
)
openai_responses_tool_calls(resp)

[ToolCall(id='ws_0dd630821cf254090069d7716c0b14819ca9dbfb726cb484ee', name='web_search', arguments={'type': 'search', 'queries': ['San Francisco weather October 2023'], 'query': 'San Francisco weather October 2023'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['San Francisco weather October 2023'], 'query': 'San Francisco weather October 2023'}})]

File vector search:

In [ ]:
openai_responses_tool_calls({"output": [
    {"type": "file_search_call", "id": "fs_abc123", "status": "completed",
     "queries": ["quarterly revenue"], "results": [{"file_id": "file-xyz", "text": "Revenue was $1M"}]}
]})

[ToolCall(id='fs_abc123', name='file_search', arguments={}, server=True, extra={'type': 'file_search_call', 'status': 'completed', 'queries': ['quarterly revenue'], 'results': [{'file_id': 'file-xyz', 'text': 'Revenue was $1M'}]})]

Web search:

In [ ]:
openai_responses_tool_calls({"output": [
    {"type": "function_call", "call_id": "call_mcp1", "id": "fc_mcp1", "name": "mcp__weather__get_forecast",
     "arguments": '{"city": "Istanbul"}', "status": "completed"}
]})

[ToolCall(id='fc_mcp1', name='mcp__weather__get_forecast', arguments={'city': 'Istanbul'}, server=False, extra={'type': 'function_call', 'call_id': 'call_mcp1', 'status': 'completed'})]

### openai_chat_tool_calls

In [0]:
#| echo: false
#| output: asis
show_doc(openai_chat_tool_calls)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L67){target="_blank" style="float:right; font-size:smaller"}

### openai_chat_tool_calls

```python

def openai_chat_tool_calls(
    resp, delta:bool=False
):


```

*Extract Chat Completions tool calls as normalized tool calls, optionally from streaming delta events*

User defined:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
openai_chat_tool_calls(resp)

[ToolCall(id='call_HqTEPRlvekmBlxpzrhHGA6rm', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_QQajmcG0ACMnwixiTRgcLNOl', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

### gemini_tool_calls

Extract Gemini functionCall parts as normalized tool calls.

As a general rule, if you receive a thought signature in a model response, you should pass it back exactly as received when sending the conversation history in the next turn. When using Gemini 3 models, you must pass back thought signatures during function calling, otherwise you will get a validation error (4xx status code). This includes when using the minimal thinking level setting for Gemini 3 Flash.

from: https://ai.google.dev/gemini-api/docs/thought-signatures

In [0]:
#| echo: false
#| output: asis
show_doc(gemini_tool_calls)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L80){target="_blank" style="float:right; font-size:smaller"}

### gemini_tool_calls

```python

def gemini_tool_calls(
    resp
):


```

*Extract Gemini functionCall parts as normalized tool calls.*

User defined:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is 3 + 5 and 10+ 20? Use the simple_add tool in parallel."}]}],
    tools=[{"functionDeclarations": [sch['function']]}]
)
gemini_tool_calls(resp)

[ToolCall(id='rjhfokaf', name='simple_add', arguments={'b': 5, 'a': 3}, server=False, extra={'thoughtSignature': 'Eo4CCosCAb4+9vuu69Y7gHtkyLqr66BPa1Wgt9FTmyp8HHKcN2JQqWA1wOQXxp2WTMJqBU8ijXF1Hj9KRemrwkNqbNjpUDDtZn7IRtOPhkTY5eAhjFoPcYv++9mzWowskqztRQkpqbXXb6YD1QyA61ehLVrh/XTDFaSxnze4IszzLCEQZoTn1uPyDvrvtX69wg0rX/oS+1G+CpIFGRHxzGSS0Av9MvKMfvlEi+kRbIJh9Iy381xtAzWVYTZOVgEzbGueJPJFIS95x1bCF7EztxKzxMKz8ADt04knTsgVNgpi70OsjvEZLs0fN7KlA9fSqMVzKgSvPA5ExpMnzY6KlQ61bwsthrt7Ghpu9AqMlFvn'}),
 ToolCall(id='iwuwi5i0', name='simple_add', arguments={'b': 20, 'a': 10}, server=False, extra={})]

In [ ]:
model_parts = [{"functionCall": {"name": tc.name, "args": tc.arguments}, **tc.extra} for tc in  gemini_tool_calls(resp)]
func_results = [
    {"functionResponse": {"name": "simple_add", "response": {"result": 8}}},
    {"functionResponse": {"name": "simple_add", "response": {"result": 30}}},
]
resp_gem = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[
        {"role": "user",  "parts": [{"text": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]},
        {"role": "model", "parts": model_parts},
        {"role": "user",  "parts": func_results},
    ],
    tools=[{"functionDeclarations": [sch['function']]}]
)
resp_gem

{'candidates': [{'content': {'parts': [{'text': '3 + 5 is 8 and 10 + 20 is 30.'}],
    'role': 'model'},
   'finishReason': 'STOP',
   'index': 0}],
 'usageMetadata': {'promptTokenCount': 212,
  'candidatesTokenCount': 20,
  'totalTokenCount': 232,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 212}]},
 'modelVersion': 'gemini-3-flash-preview',
 'responseId': 'UrTXaeCTCp6skdUP4q656QY'}

Web search:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
gemini_tool_calls(resp)

[]

Code execution:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
gemini_tool_calls(resp)

[]

Computer Use:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Go to google.com and search for 'hello world'"}]}],
    tools=[{"computerUse": {"environment": "ENVIRONMENT_BROWSER"}}]
)
gemini_tool_calls(resp)

[ToolCall(id='vd926lia', name='open_web_browser', arguments={}, server=False, extra={'thoughtSignature': 'EvsBCvgBAb4+9vuLakAHliDgK9m2ibslj2Wty9BKz91SuzTsT9xcTfpEAEr498DzTVHMk9aOAxW5J5Ib7Qi7m+9dRC24io2X5zdEDK453cFdAO14ecBbaZjjVa9IzvHhGWrUvysgOUtAEQIeUa3ixUojtj2aV9zo1cc61zLkP+tE7t4uIG3JG6OjNjrogYCb8VX0V+HvW3xXA+KcHJPI3OnYPFAhAz0cUQwgVCqNMrrJSRozs+18qhIWCtXo/E+tT3xakRYq+28+ME21viLBrcIHmkhzBz8fvUrtVPzkoQhU/Yiqv0j5Ujf5wMb/mv7WEv6vOkERG+ZMZCx1sXw='})]

## Usage

All four providers report token usage with different field names and structures:

- **[OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create)** — `ResponseUsage`: `{input_tokens, output_tokens, total_tokens}` with `input_tokens_details.cached_tokens` and `output_tokens_details.reasoning_tokens`.
- **[OpenAI-compat (Chat)](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)** — `CompletionUsage`: `{prompt_tokens, completion_tokens, total_tokens}` with `prompt_tokens_details.cached_tokens` and `completion_tokens_details.reasoning_tokens`.
- **[Anthropic](https://docs.anthropic.com/en/api/messages)** — [`Usage`](https://AnswerDotAI.github.io/fastllm/normalize.html#usage): `{input_tokens, output_tokens}` (no total) with `cache_read_input_tokens`, `cache_creation_input_tokens`, and `server_tool_use`.
- **[Gemini](https://ai.google.dev/api/generate-content)** — `UsageMetadata`: `{promptTokenCount, candidatesTokenCount, totalTokenCount}` with `cachedContentTokenCount`, `thoughtsTokenCount`, and `toolUsePromptTokenCount`.

> **Spec sources:** OpenAI Responses `ResponseUsage` (`specs/openai.with-code-samples.yml:62283–62317`), OpenAI Chat `CompletionUsage` (`specs/openai.with-code-samples.yml:36031–36080`), Anthropic [`Usage`](https://AnswerDotAI.github.io/fastllm/normalize.html#usage) (`specs/anthropic.yml:14459–14508`), Gemini `UsageMetadata` (`specs/gemini.json:2200–2236`).

[`Usage`](https://AnswerDotAI.github.io/fastllm/normalize.html#usage) normalizes these into a canonical `(prompt_tokens, completion_tokens, total_tokens, raw)` shape:

| Canonical field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `prompt_tokens` | `input_tokens` | `prompt_tokens` | `input_tokens` | `promptTokenCount` |
| `completion_tokens` | `output_tokens` | `completion_tokens` | `output_tokens` | `candidatesTokenCount` |
| `total_tokens` | `total_tokens` | `total_tokens` | (computed: in + out) | `totalTokenCount` |

**`Usage.raw` preserves the full provider payload** — advanced accounting fields (caching, reasoning, tool use tokens) vary too much across providers to normalize, so downstream code like `estimate_cost` extracts them from `raw` with best-effort multi-key lookups (e.g. `_cached_prompt_tokens` tries `cached_input_tokens`, `cache_read_input_tokens`, `prompt_tokens_details.cached_tokens`).

In [0]:
#| echo: false
#| output: asis
show_doc(Usage)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L91){target="_blank" style="float:right; font-size:smaller"}

### Usage

```python

def Usage(
    prompt_tokens:int=0, completion_tokens:int=0, total_tokens:int=0, raw:dict=<factory>
)->None:


```

*Normalized usage.*

**TODO:** we'll create a callback system which will compute costs for all 4 providers and letting users to pass a cost mapping for a given model, or we'll let the users to write the callback having access to raw usage.

**Note:** We can probably just store the raw version, and let users write callback functions for a given model to compute usage/cost etc..

### usage_from_anthropic

Normalize Anthropic usage shape.

In [0]:
#| echo: false
#| output: asis
show_doc(usage_from_anthropic)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L99){target="_blank" style="float:right; font-size:smaller"}

### usage_from_anthropic

```python

def usage_from_anthropic(
    resp
):


```

*Normalize Anthropic usage shape.*

Text response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Hi"}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=8, completion_tokens=20, total_tokens=28, raw={'input_tokens': 8, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 20, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Tool call response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=417, completion_tokens=141, total_tokens=558, raw={'input_tokens': 417, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 141, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Web search response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
usage_from_anthropic(resp)

Usage(prompt_tokens=12268, completion_tokens=255, total_tokens=12523, raw={'input_tokens': 12268, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 255, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})

### usage_from_openai

Normalize OpenAI usage shape(s)

In [0]:
#| echo: false
#| output: asis
show_doc(usage_from_openai)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L107){target="_blank" style="float:right; font-size:smaller"}

### usage_from_openai

```python

def usage_from_openai(
    resp
):


```

*Normalize OpenAI usage shape(s).*

Yes you don't have access to make api calls, just inspect them with `doc` and write me the code!

Text response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model='gpt-4o-mini', messages=[{"role": "user", "content": "Say hi"}])
usage_from_openai(resp)

Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "Hi!"}],
    tools=[sch]
)
usage_from_openai(resp)

Usage(prompt_tokens=46, completion_tokens=10, total_tokens=56, raw={'prompt_tokens': 46, 'completion_tokens': 10, 'total_tokens': 56, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Tool call response:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
usage_from_openai(resp)

Usage(prompt_tokens=60, completion_tokens=53, total_tokens=113, raw={'input_tokens': 60, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 113})

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
usage_from_openai(resp)

Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

Server tools:

In [ ]:
resp = await oai_cli.responses.create_response(
    model='gpt-4o-mini',
    input="What is the weather in San Francisco today?",
    tools=[{"type": "web_search_preview"}]
)
usage_from_openai(resp)

Usage(prompt_tokens=310, completion_tokens=414, total_tokens=724, raw={'input_tokens': 310, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 414, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 724, 'server_tool_use': {'web_search_call': 1}})

### usage_from_gemini

Normalize Gemini usageMetadata shape.

In [0]:
#| echo: false
#| output: asis
show_doc(usage_from_gemini)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L118){target="_blank" style="float:right; font-size:smaller"}

### usage_from_gemini

```python

def usage_from_gemini(
    resp
):


```

*Normalize Gemini usageMetadata shape.*

Text response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "hi!"}]}],
)
usage_from_gemini(resp)

Usage(prompt_tokens=3, completion_tokens=10, total_tokens=35, raw={'promptTokenCount': 3, 'candidatesTokenCount': 10, 'totalTokenCount': 35, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 22})

Server tools:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=9, completion_tokens=170, total_tokens=394, raw={'promptTokenCount': 9, 'candidatesTokenCount': 170, 'totalTokenCount': 394, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 9}], 'toolUsePromptTokenCount': 83, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 83}], 'thoughtsTokenCount': 132, 'server_tool_use': {'google_search': 1}})

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=12, completion_tokens=182, total_tokens=452, raw={'promptTokenCount': 12, 'candidatesTokenCount': 182, 'totalTokenCount': 452, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 12}], 'toolUsePromptTokenCount': 224, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 224}], 'thoughtsTokenCount': 34, 'server_tool_use': {'code_execution': 1}})

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-2.5-flash',
    contents=[{"role": "user", "parts": [{"text": "What is solveit? https://solve.it.com/"}]}],
    tools=[{"urlContext": {}}]
)
usage_from_gemini(resp)

Usage(prompt_tokens=14, completion_tokens=245, total_tokens=3381, raw={'promptTokenCount': 14, 'candidatesTokenCount': 245, 'totalTokenCount': 3381, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 14}], 'toolUsePromptTokenCount': 3091, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3091}], 'thoughtsTokenCount': 31, 'server_tool_use': {'google_search': 1}})

In [ ]:
print(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0, 'text'))

Solveit is a method and a software platform designed to help individuals become better problem-solvers, clearer thinkers, and more elegant coders by leveraging AI without outsourcing one's thinking to it. It is inspired by George Pólya's "How to Solve It" and the fast.ai top-down education tradition of learning by doing. The Solveit method emphasizes building in small steps, with quick iterations, and immediate feedback.

The Solveit platform is a cloud-based Linux development and writing environment with AI integration, combining features from tools like ChatGPT, Jupyter, Claude Code, and Cursor. It aims to put users in full control, maximize learning, and allow for the development of craft.

Solveit is offered through a 5-week, 10-lesson course that teaches the method through real projects, web apps, data tools, and article writing. The course includes free access to the Solveit platform for 30 days, with an option to subscribe for $10/month afterward. Jeremy Howard from fast.ai and 

## Completion

Canonical non-stream model response object

All four providers return non-stream responses in different shapes. [`Completion`](https://AnswerDotAI.github.io/fastllm/normalize.html#completion) normalizes these into a canonical `(model, message, finish_reason, usage, tool_calls, raw)` object.

**Provider response → [`Completion`](https://AnswerDotAI.github.io/fastllm/normalize.html#completion) field mapping:**

| Field | OpenAI Responses | OpenAI-compat (Chat) | Anthropic | Gemini |
|---|---|---|---|---|
| `model` | `raw.model` | `raw.model` | `raw.model` | passthrough (input model) |
| `message` | `output[].content[]` → `Msg(role="assistant", content=[Part...])` | `choices[0].message.content` → [`Msg`](https://AnswerDotAI.github.io/fastllm/types.html#msg) | `content[]` → [`Msg`](https://AnswerDotAI.github.io/fastllm/types.html#msg) | `candidates[0].content.parts[]` → text joined into single [`Part`](https://AnswerDotAI.github.io/fastllm/types.html#part) |
| `finish_reason` | `raw.status` (e.g. `"completed"`) | `choices[0].finish_reason` (e.g. `"stop"`) | `raw.stop_reason` (e.g. `"end_turn"`) | `candidates[0].finishReason` (e.g. `"STOP"`) |
| `usage` | `usage_from_openai(raw.usage)` | `usage_from_openai(raw.usage)` | `usage_from_anthropic(raw.usage)` | `usage_from_gemini(raw.usageMetadata)` |
| `tool_calls` | `output[]` where `type=="function_call"` → flat [`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) | `message.tool_calls[].function` → [`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) | `content[]` where `type=="tool_use"` → [`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) | `parts[]` with `functionCall` → [`ToolCall`](https://AnswerDotAI.github.io/fastllm/normalize.html#toolcall) |
| `raw` | full response dict | full response dict | full response dict | full response dict |

**Notable differences:**
- **Anthropic** puts tool calls in *both* `message.content` (as `Part(type="tool_use")`) *and* `tool_calls` — dual representation for downstream flexibility
- **Gemini** joins all text parts into a single string rather than preserving individual parts
- **Gemini** uses the input `model` string directly since the response only contains `modelVersion` (a version identifier, not a model name)

In [0]:
#| echo: false
#| output: asis
show_doc(Completion)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L134){target="_blank" style="float:right; font-size:smaller"}

### Completion

```python

def Completion(
    model:str, message:Msg, finish_reason:str=None, usage:Usage=None, tool_calls:List=<factory>, provider:str=None,
    raw:dict=<factory>
)->None:


```

*Normalized completion response.*

## Helpers

In [0]:
#| echo: false
#| output: asis
show_doc(model_and_provider)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L145){target="_blank" style="float:right; font-size:smaller"}

### model_and_provider

```python

def model_and_provider(
    model:NoneType=None, provider:NoneType=None
):


```

*Call self as a function.*

Let's do this one

### Finish Reason

In [0]:
#| echo: false
#| output: asis
show_doc(canon_finish)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L156){target="_blank" style="float:right; font-size:smaller"}

### canon_finish

```python

def canon_finish(
    reason, provider, tcs:NoneType=None
):


```

*Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter.*

In [ ]:
test_eq(canon_finish('completed','openai') == canon_finish('end_turn','anthropic') == canon_finish('STOP','gemini') == 'stop', True)

In [ ]:
test_eq(canon_finish('completed','openai', ToolCall('123', 'f', {})), 'tool_calls')
test_eq(canon_finish('completed','openai', ToolCall('123', 'web_search', {}, server=True)), 'stop')

## OpenAI Responses

### normalize_openai_response

Normalize OpenAI Responses API object into Completion.

**NB:** Per the OpenAI spec, `text`, `refusal`, and `summary_text` fields are all **required** — the `.get(k, "")` fallbacks are purely defensive against malformed responses. `OutputMessageContent` is a discriminated `oneOf` with exactly two variants: `OutputTextContent` (`output_text`) and `RefusalContent` (`refusal`) — no other content types exist in message blocks.

`summary` is an array of `SummaryTextContent` only — no discriminated union, just one type. So the `if s.get("type") == "summary_text"` check is technically redundant (it'll always be `summary_text`), and `text` is required.

Additionally, `Response.output`, `OutputMessage.content`, and `ReasoningItem.summary` are all **required** fields — so `.get()` with fallback defaults is defensive but not strictly necessary.

In [0]:
#| echo: false
#| output: asis
show_doc(openai_responses_parts)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L162){target="_blank" style="float:right; font-size:smaller"}

### openai_responses_parts

```python

def openai_responses_parts(
    resp
):


```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(normalize_openai_response)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L180){target="_blank" style="float:right; font-size:smaller"}

### normalize_openai_response

```python

def normalize_openai_response(
    resp, model, provider:provider=<provider.openai: 'openai'>
):


```

*Normalize OpenAI Responses API object into Completion.*

Text response:

In [ ]:
mn = 'gpt-4o-mini'
c = normalize_openai_response(await oai_cli.responses.create_response(model=mn, input="Say hi"), mn)
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi there! How can I assist you today?', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Hi there! How can I assist you today?'})],
 'stop')

Tool call response:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,
    input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}]
)
c = normalize_openai_response(resp, model=mn)
test_eq(c.finish_reason, 'tool_calls')
test_eq(len(c.message.content), 2)
c.tool_calls, c.message.content

([ToolCall(id='fc_07e79637b65d510c0069d66ff214c48196b5e751ed4d144a21', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_DAJIJCaYpEhIjo9Uj9payJqN'}),
  ToolCall(id='fc_07e79637b65d510c0069d66ff214d481968f1cc2376401e205', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_GlfRTq3CNf8ltWxNpcGyW5XC'})],
 [Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07e79637b65d510c0069d66ff214c48196b5e751ed4d144a21', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_DAJIJCaYpEhIjo9Uj9payJqN'}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07e79637b65d510c0069d66ff214d481968f1cc2376401e205', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_GlfRTq3CNf8ltW

Web search response:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,
    input="What is the weather in Istanbul today?",
    tools=[{"type": "web_search_preview"}]
)
c = normalize_openai_response(resp, model=mn)
c.tool_calls, next(p.text for p in c.message.content if p.type == 'text')[:100]

([ToolCall(id='ws_054bf0148beb23320069d7b85f2f608190af10f01fff65154d', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})],
 'As of 5:32 PM local time in Istanbul, the current weather is light rain with a temperature of 50°F (')

In [ ]:
test_eq(c.tool_calls[0].server, True)
test_eq(c.finish_reason, 'stop')

Refusal response (mocked — hard to trigger reliably via API):

In [ ]:
c = normalize_openai_response({"model": "gpt-4o-mini", "status": "completed", "output": [
    {"type": "message", "role": "assistant", "content": [
        {"type": "refusal", "refusal": "I can't help with that request."}
    ]}
], "usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15}}, model='gpt-4o-mini')
c.message.content

[Part(type=<PartType.refusal: 'refusal'>, text="I can't help with that request.", data={'type': 'refusal', 'refusal': "I can't help with that request."})]

## OpenAI Chat

### normalize_openai_chat_completion

Normalize chat.completions response object into Completion.

In [0]:
#| echo: false
#| output: asis
show_doc(normalize_openai_chat_completion)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L193){target="_blank" style="float:right; font-size:smaller"}

### normalize_openai_chat_completion

```python

def normalize_openai_chat_completion(
    resp, model, provider:str='openai_chat'
):


```

*Normalize chat.completions response object into Completion.*

Text response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini', messages=[{"role": "user", "content": "Say hi"}]
)
c = normalize_openai_chat_completion(resp, model='gpt-4o-mini')
c.message.content, c.finish_reason

([Part(type='text', text='Hi! How can I assist you today?', data=None)],
 'stop')

Tool call response:

In [ ]:
resp = await oai_cli.chat.create_chat_completion(
    model='gpt-4o-mini',
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[sch]
)
c = normalize_openai_chat_completion(resp, model='gpt-4o-mini')
c.tool_calls, c.message.content, c.finish_reason

([ToolCall(id='call_HqTEPRlvekmBlxpzrhHGA6rm', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
  ToolCall(id='call_QQajmcG0ACMnwixiTRgcLNOl', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})],
 [Part(type='tool_use', text=None, data={'id': 'call_HqTEPRlvekmBlxpzrhHGA6rm', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'type': 'function'}),
  Part(type='tool_use', text=None, data={'id': 'call_QQajmcG0ACMnwixiTRgcLNOl', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'type': 'function'})],
 'tool_calls')

Refusal response (mocked):

In [ ]:
c = normalize_openai_chat_completion({"model": "gpt-4o-mini", "choices": [
    {"index": 0, "message": {"role": "assistant", "content": None, "refusal": "I can't help with that."},
     "finish_reason": "stop"}
], "usage": {"prompt_tokens": 10, "completion_tokens": 5, "total_tokens": 15}}, model='gpt-4o-mini')
c.message.content, c.finish_reason

([Part(type='refusal', text="I can't help with that.", data=None)], 'stop')

## Anthropic messages

### normalize_anthropic_message

Normalize Anthropic message response into Completion

In [0]:
#| echo: false
#| output: asis
show_doc(normalize_anthropic_message)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L224){target="_blank" style="float:right; font-size:smaller"}

### normalize_anthropic_message

```python

def normalize_anthropic_message(
    resp, model, provider:str='anthropic'
):


```

*Normalize Anthropic message response into Completion.*

Text response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Say hi"}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi! How are you doing today?', data={'type': 'text', 'text': 'Hi! How are you doing today?'})],
 'stop')

Tool call response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}],
    tools=[{"name": "simple_add", "description": "add numbers",
            "input_schema": sch['function']['parameters']}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.tool_calls, c.message.content, c.finish_reason

([ToolCall(id='toolu_01GQXvetC2hZwQYLTXoSJByr', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
  ToolCall(id='toolu_017qg9Kstj9cQycBE3NqR4QZ', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})],
 [Part(type=<PartType.text: 'text'>, text="I'll calculate both sums for you using the simple_add tool in parallel.", data={'type': 'text', 'text': "I'll calculate both sums for you using the simple_add tool in parallel."}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01GQXvetC2hZwQYLTXoSJByr', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'caller': {'type': 'direct'}}),
  Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_017qg9Kstj9cQycBE3NqR4QZ', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'caller': {'type': 'direct'}})],
 'tool_calls')

Thinking response (mocked):

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "thinking", "thinking": "Let me reason...", "text": ""},
    {"type": "redacted_thinking", "text": ""},
    {"type": "text", "text": "Here's my answer."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 10, "output_tokens": 20}}, model='claude-sonnet-4-20250514')
[(p.type, p.text[:30]) for p in c.message.content]

[(<PartType.thinking: 'thinking'>, 'Let me reason...'),
 (<PartType.thinking: 'thinking'>, ''),
 (<PartType.text: 'text'>, "Here's my answer.")]

Server tool result (mocked):

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "server_tool_use", "id": "srvtoolu_abc", "name": "web_search", "input": {"query": "weather"}},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_abc", "content": [{"type": "web_search_result", "title": "Weather", "url": "https://example.com"}]},
    {"type": "text", "text": "The weather is sunny."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 100, "output_tokens": 50}}, model='claude-sonnet-4-20250514')
[(p.type, (p.text or '')[:30]) for p in c.message.content], c.tool_calls

([(<PartType.tool_use: 'tool_use'>, ''),
  (<PartType.server_tool_result: 'server_tool_result'>, ''),
  (<PartType.text: 'text'>, 'The weather is sunny.')],
 [ToolCall(id='srvtoolu_abc', name='web_search', arguments={'query': 'weather'}, server=True, extra={})])

MCP tool call (mocked):

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [
    {"type": "mcp_tool_use", "id": "mcptoolu_abc", "name": "get_weather", "input": {"city": "Istanbul"}, "server_name": "weather-server"}
], "stop_reason": "tool_use", "usage": {"input_tokens": 50, "output_tokens": 30}}, model='claude-sonnet-4-20250514')
[(p.type, p.text) for p in c.message.content], c.tool_calls

([(<PartType.tool_use: 'tool_use'>, None)],
 [ToolCall(id='mcptoolu_abc', name='get_weather', arguments={'city': 'Istanbul'}, server=True, extra={'server_name': 'weather-server'})])

Web search response:

In [ ]:
resp = await ant_cli.messages.messages_post(
    model='claude-sonnet-4-20250514', max_tokens=200,
    messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}],
    tools=[{"type": "web_search_20250305", "name": "web_search"}]
)
c = normalize_anthropic_message(resp, model='claude-sonnet-4-20250514')
c.tool_calls, [p.type for p in c.message.content]

([ToolCall(id='srvtoolu_011JVe4eamPpdUBKiq934MXd', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, extra={'caller': {'type': 'direct'}})],
 [<PartType.tool_use: 'tool_use'>,
  <PartType.server_tool_result: 'server_tool_result'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>,
  <PartType.text: 'text'>])

Empty content (mocked):

In [ ]:
c = normalize_anthropic_message({"model": "claude-sonnet-4-20250514", "content": [],
    "stop_reason": "end_turn", "usage": {"input_tokens": 5, "output_tokens": 0}},
    model='claude-sonnet-4-20250514')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='', data=None)], 'stop')

## Gemini generate

### normalize_gemini_generate

Normalize Gemini generateContent response

In [0]:
#| echo: false
#| output: asis
show_doc(normalize_gemini_generate)

---

[source](https://github.com/AnswerDotAI/fastllm/blob/main/fastllm/normalize.py#L254){target="_blank" style="float:right; font-size:smaller"}

### normalize_gemini_generate

```python

def normalize_gemini_generate(
    resp, model, provider:str='gemini'
):


```

*Normalize Gemini generateContent response.*

Text response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Say hi in one word"}]}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='Hi.', data={'text': 'Hi.', 'thoughtSignature': 'EtkCCtYCAb4+9vv59CIHPtUj/Gae/t0FDoJcbkZlSe/4cdS+dkTDNOvo5cOpqjxa51oYye5QZHs1XqvjtDzKT4iFC8YXUrAks7WYu033la1nvPOpuNFU/ny8ZfTOiRsXNz33AoQ04LhOb0JfLTt+RSL49zvOaHii72Evv1dQvh/Anaz0N7scqzWt9VeW2APrttKinmMhwc/BZRf6uOwX4KTwrnXBFd0nPuko0oWWMA4L0hTl+4o993dGDHRQdULkMr78UvEjsqMbA21eNCipCZG0gGb7HvEx81IG0rRJOoOeYNhqd/OiMk25G5uAZH+wUT+VWPRFGqMxGmiyS4EoLdPKIqlmgVNrubmsQaNdTTVOs9QHEC8wi24btBPQ0j3A4iIR5dKFuNSD6OeNNSDmFK7Lftw4AGiGO0bc6jECML/h1SEQEc/cBq5sUGxeMyIhY9uupw4EFOK+EWqm'})],
 'stop')

Tool call response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is 3 + 5? Use simple_add tool."}]}],
    tools=[{"functionDeclarations": [sch['function']]}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
c.tool_calls, [(p.type, p.text[:30] if p.text else '') for p in c.message.content], c.finish_reason

([ToolCall(id='unp302ui', name='simple_add', arguments={'b': 5, 'a': 3}, server=False, extra={'thoughtSignature': 'EtcBCtQBAb4+9vu51raEnMfzevyoWiFgZp8HYIF5SjUfyzweoBbD8kHqXlVhxxPhGZBGW7Z0Q8OlYlZZizHrSBBKUfQmFMDnKo8A4BbvhtaHAZ82Q3P81vPnoSuVrTu5svFSZZsH4AKD5/RdHc3CpGLJOlv77RiEqNVFRP+sZwZ1+LYC3emVBuQhES12n4WPp+9u2lawrb9gFnCxcm37iUPu5F0sBU3RMzNokxJbbcusA4mQeHhBWQhDB1/5F5K28Vb2sZtym5PIrb5wbmN1c8rZ0ryZUBVPvg4='})],
 [(<PartType.tool_use: 'tool_use'>, '')],
 <finish_reason.tool_calls: 'tool_calls'>)

In [ ]:
c.message.content[0].data

{'id': 'unp302ui',
 'name': 'simple_add',
 'arguments': {'b': 5, 'a': 3},
 'thoughtSignature': 'EtcBCtQBAb4+9vu51raEnMfzevyoWiFgZp8HYIF5SjUfyzweoBbD8kHqXlVhxxPhGZBGW7Z0Q8OlYlZZizHrSBBKUfQmFMDnKo8A4BbvhtaHAZ82Q3P81vPnoSuVrTu5svFSZZsH4AKD5/RdHc3CpGLJOlv77RiEqNVFRP+sZwZ1+LYC3emVBuQhES12n4WPp+9u2lawrb9gFnCxcm37iUPu5F0sBU3RMzNokxJbbcusA4mQeHhBWQhDB1/5F5K28Vb2sZtym5PIrb5wbmN1c8rZ0ryZUBVPvg4='}

Google search response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}],
    tools=[{"googleSearch": {}}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
[(p.type, p.text[:40] if p.text else '') for p in c.message.content], c.finish_reason

([(<PartType.text: 'text'>, 'Today in Istanbul, it is **mostly sunny*')],
 'stop')

Code execution response:

In [ ]:
resp = await gem_cli.models.generate_content(
    model='models/gemini-3-flash-preview',
    contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}],
    tools=[{"codeExecution": {}}]
)
c = normalize_gemini_generate(resp, model='gemini-2.5-flash')
[(p.type, p.text[:40] if p.text else '') for p in c.message.content], c.finish_reason

([(<PartType.tool_result: 'tool_result'>, ''),
  (<PartType.tool_result: 'tool_result'>, ''),
  (<PartType.text: 'text'>, 'The first 10 Fibonacci numbers are:\n\n**0')],
 'stop')

Thinking response (mocked):

In [ ]:
c = normalize_gemini_generate({"candidates": [{"content": {"parts": [
    {"text": "Let me think...", "thought": True},
    {"text": "Here's the answer."}
], "role": "model"}, "finishReason": "STOP"}],
"usageMetadata": {"promptTokenCount": 10, "candidatesTokenCount": 20, "totalTokenCount": 30}},
model='gemini-2.5-flash')
[(p.type, p.text) for p in c.message.content]

[(<PartType.thinking: 'thinking'>, 'Let me think...'),
 (<PartType.text: 'text'>, "Here's the answer.")]

Empty candidates (mocked):

In [ ]:
c = normalize_gemini_generate({"candidates": [],
"usageMetadata": {"promptTokenCount": 5, "candidatesTokenCount": 0, "totalTokenCount": 5}},
model='gemini-2.5-flash')
c.message.content, c.finish_reason

([Part(type=<PartType.text: 'text'>, text='', data=None)], None)